In [9]:
import pandas as pd
import os
import csv
import re
import ftfy
import unicodedata
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.processors import TemplateProcessing
from tokenizers import Tokenizer
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from sklearn.model_selection import train_test_split
import time

torch.manual_seed(42)

BATCH_SIZE     = 64
CONTEXT_SIZE   = 256
MAX_ITERS      = 3000
EVAL_INTERVAL  = 100
LR_RATE        = 3e-4
EVAL_ITERS     = 10
device = 'cuda' if torch.cuda.is_available() else 'cpu'



In [10]:
tokenizer = Tokenizer.from_file(
    r'..\data\processed\tokenizer.json'
)
tokenizer.no_padding()


In [11]:
EOS_TOKEN_ID = tokenizer.token_to_id("[/S]")
EOS_TOKEN_ID

1

In [12]:
@torch.no_grad()
def estimate_loss(context_size):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(context_size=context_size,data=split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class Head(nn.Module):
    def __init__(self, emb_dim, context_size, head_size, dropout):
        super().__init__()
        self.key   = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))

        self.dropout = nn.Dropout(dropout)

        
    def forward(self, x, attn_mask=None):
        B, T, E = x.shape
        k = self.key(x)    # (B, T, H)
        q = self.query(x)  # (B, T, H)

        weights = q @ k.transpose(-2, -1) * E**-0.5 # (B, T, H) @ (B, H, T) -> (B, T, T)
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        
        if attn_mask is not None:
            pad_mask = attn_mask[:, None, :].to(weights.device)  # (B, 1, T)
            weights = weights.masked_fill(~pad_mask, float('-inf'))
        
        weights = F.softmax(weights, dim=-1)        # (B, T, T)
        weights = self.dropout(weights)
        
        v = self.value(x)  # (B, T, E)
        out = weights @ v  # (B, T, T) @ (B, T, E) -> (B, T, E)
        return out
    
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, emb_dim, context_size, head_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(emb_dim, context_size, head_size, dropout) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(num_heads * head_size, emb_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, attn_mask=None):
        out = torch.cat([h(x, attn_mask=attn_mask) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

        
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, emb_dim, context_size, num_heads, dropout):
        super().__init__()
        head_size = emb_dim // num_heads
        self.sa   = MultiHeadAttention(num_heads, emb_dim, context_size, head_size, dropout)
        self.ffwd = FeedForward(emb_dim, dropout)
        self.ln1  = nn.LayerNorm(emb_dim)
        self.ln2  = nn.LayerNorm(emb_dim)
        
    def forward(self, x, attn_mask=None):
        # x + layer because of residual connections
        # deviation from paper -> pre-norm (need to test)
        x = x + self.sa(self.ln1(x), attn_mask=attn_mask)
        x = x + self.ffwd(self.ln2(x))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, context_size, num_att_heads, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.context_size = context_size
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(context_size, emb_dim)

        # self.sa_heads  = MultiHeadAttention(num_att_heads, emb_dim, context_size, head_size=emb_dim//4)

        # self.ffwd = FeedForward(emb_dim)

        self.blocks = nn.ModuleList([
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_emb = self.token_embedding(idx) # (B, T, E)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device)) # (T, E)
        x = token_emb + pos_emb  # (B, T, E)
        # x = self.sa_heads(x)     # (B, T, E)
        # x = self.ffwd(x)         # (B, T, E)

         # Create attention mask: True = keep, False = pad
        # attn_mask = (idx != PAD_TOKEN_ID)  # shape (B, T)
        attn_mask = None
        
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask)
        x = self.ln_f(x)
        
        logits = self.lm_head(x) # (B, T, V)
        
        loss = None
        if targets is not None:
            # Flatten both for cross-entropy
            logits = logits.view(B * T, self.vocab_size)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets, ignore_index=PAD_TOKEN_ID)
        
        return logits, loss

    def generate(self, idx, max_new_tokens=256):
        for _ in range(max_new_tokens):    
            idx_cond = idx[:, -self.context_size:] # when max_new_tokens > context size
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if idx_next == EOS_TOKEN_ID:
                break
            
        return idx


In [13]:
context_size = 256
model = TransformerDecoder(vocab_size=30000, emb_dim=512, context_size=context_size, num_att_heads=6, dropout=0.2)
model = model.to(device)


In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_RATE)

In [7]:
checkpoint_path = r'..\data\model_checkpoints'


checkpoint = torch.load('..\data\model_checkpoints\checkpoint_9800.pt', map_location='cuda')  # or 'cuda' if available

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_iteration = checkpoint['iteration']
loss_value = checkpoint['loss']

print(f"✅ Loaded checkpoint from iteration {start_iteration}, last loss = {loss_value:.4f}")

model.eval()  

✅ Loaded checkpoint from iteration 9800, last loss = 2.9437


TransformerDecoder(
  (token_embedding): Embedding(30000, 512)
  (position_embedding): Embedding(256, 512)
  (blocks): ModuleList(
    (0-5): 6 x Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=512, out_features=85, bias=False)
            (query): Linear(in_features=512, out_features=85, bias=False)
            (value): Linear(in_features=512, out_features=85, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=510, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): ReLU()
          (2): Linear(in_features=2048, out_features=512, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=

In [9]:
x = "tell me a joke about man, woman, friend [JOKE]"
tok_x = tokenizer.encode(x)
tok_ids = tok_x.ids[:-1]
torch.tensor([tok_ids], dtype=torch.long, device=device)
idx = torch.tensor([tok_ids], dtype=torch.long, device=device)
res = tokenizer.decode(model.generate(idx)[0].tolist(), skip_special_tokens=False)
res

'[S] tell me a joke about man, woman, friend [JOKE] poor asian woman decided to go to a german women and visit a friend . the men noticed the chinese woman was telling the italian men that the men would hire a man to be pretty confused , by looking for respect he would for her because they would be the spice ! the man replies " i\'m sorry , i had a friend named wendy . " the chinese woman walks out of the room and asks , " so how did you get the drinks ? " the jewish man says , " i just had a friend named wendy because your friend jonathan named wendy . in fact , that the women named wendy and stone masonals are for french and the chefs , but i figured the chinese gama name is wendy and pymin and some know they are " wendy im the one whose names . " her friend comes to visit what she would name an mopractor ! so he gives her a can of custard ... asks whose " name wendy wendy , wendy king " . perplexed , the jewish man sitting at the kitchen angrily opens his mouth , hi wendy heard wend

In [16]:
# dummy = torch.randint(0, 30000, (1, 256))
# torch.onnx.export(model, dummy, "joke_transformer.onnx")
